[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gforconi/UTNIA2025/blob/main/NLP_4_langchain_agentes.ipynb)


# ¿Qué es el parser ReAct de LangChain?

En **LangChain**, el término **ReAct parser** se refiere a un **analizador** que implementa el patrón **ReAct (Reason + Act)**.

---

## 🔹 Contexto ReAct
ReAct es un enfoque presentado en un paper de Google/Stanford (2022) donde un modelo de lenguaje alterna entre **razonar paso a paso** y **ejecutar acciones**.

1. El LLM genera un *pensamiento* (por ejemplo: "Pienso que debo buscar en la base de datos").
2. El LLM propone una *acción* (por ejemplo: "Ejecutar: SELECT ...").
3. Se recibe una observación de esa acción (el resultado de la consulta).
4. El LLM continúa con otro *razonamiento*, y así sucesivamente.

En LangChain, este patrón se implementa con los **Agentes ReAct**.

---

## 🔹 ¿Qué hace el Parser ReAct?
El **ReAct parser** es el componente que:
- **Lee la salida del LLM** (texto libre).
- **Identifica** si el modelo está dando un *razonamiento* (pensamiento interno), una *acción* (qué herramienta llamar), o una *respuesta final* al usuario.
- **Estructura esa salida en objetos** que el agente pueda entender (por ejemplo: `AgentAction`, `AgentFinish`).

En otras palabras, el parser convierte el texto generado por el LLM en un **formato estructurado** que el motor de LangChain puede usar para decidir si ejecutar una herramienta o devolver la respuesta final.

---

## 🔹 Ejemplo

### Salida del LLM:

Pensamiento: Para responder necesito buscar la población.
Acción: WikipediaSearch
Entrada: "población de Buenos Aires"

### El **ReAct parser** lo transforma en:
```python
AgentAction(
    tool="WikipediaSearch",
    tool_input="población de Buenos Aires"
 )
```

### Luego el LLM responde:
Respuesta final: La población de Buenos Aires es ~15 millones.

### El parser lo convierte en:
```python
AgentFinish(
    return_values={"output": "La población de Buenos Aires es ~15 millones."}
 )
```

---

## 📌 Resumen

El ReAct parser en LangChain interpreta la salida textual del LLM siguiendo el patrón ReAct, separando el razonamiento, la acción y la respuesta final, para que el agente pueda ejecutar herramientas o responder al usuario.

---

# El código

## Importación de librerías

In [ ]:
# !pip install -U "langchain>=0.2" langchain-openai langchain-ollama langchain-community ipywidgets

In [1]:
# --- imports base ---
# Importamos las librerías necesarias para trabajar con agentes y herramientas en LangChain.
# También importamos utilidades para manejar bases de datos y widgets opcionales para la interfaz.
import os, sqlite3, re
from typing import Literal, Optional

# Importamos funciones principales de LangChain para crear agentes y herramientas.
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate
from langchain.tools import tool

# Importamos los conectores para los modelos de lenguaje (OpenAI y Ollama).
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

# Para guardar el historial de la conversación en memoria.
from langchain.memory import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Widgets opcionales para la interfaz gráfica en notebooks.
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False


## Parámetros del agente

Si deseas trabajar con otro LLM, agrégalo como opción y, si es necesario, incorpora la API KEY o los parámetros correspondientes.

In [ ]:
# Este bloque permite ingresar la clave de OpenAI y elegir el backend (openai u ollama).
# Si hay widgets disponibles, se muestran campos interactivos para facilitar la configuración.
# La clave se guarda en las variables de entorno y el backend se puede cambiar fácilmente.
# Si no hay widgets, se usan los valores por defecto definidos en el código.
os.environ["OPENAI_API_KEY"] = 'tu_api_key_aqui'
BACKEND: Literal["openai", "ollama"] = "ollama"

if HAS_WIDGETS:
    api_key_widget = widgets.Text(
        value=os.getenv("OPENAI_API_KEY", ""),
        placeholder="Poner OPENAI_API_KEY",
        description="OpenAI Key:",
        disabled=False,
        layout=widgets.Layout(width="50%"),
    )

    backend_widget = widgets.Dropdown(
        options=["openai", "ollama"],
        value="ollama",
        description="Backend:",
        disabled=False,
    )
    
    def on_backend_change(change):
        if change["name"] == "value":
            global BACKEND
            BACKEND = change["new"]
            print("Backend cambiado a:", BACKEND)

    backend_widget.observe(on_backend_change)

    def on_api_key_change(change):
        if change["name"] == "value":
            os.environ["OPENAI_API_KEY"] = change["new"]
            print("OPENAI_API_KEY actualizada en os.environ")

    api_key_widget.observe(on_api_key_change)

    display(api_key_widget, backend_widget)

Text(value='sk-proj-ZQZHe8XDb7Bhb_xbXlhzj4jkoBEK2sM8UozQx7AumKydncpsDltHrqDMa3S8Xe06Pz0NnS8BGaT3BlbkFJYIi5EzHR…

Dropdown(description='Backend:', index=1, options=('openai', 'ollama'), value='ollama')

## Función de creación del modelo

Esta función encapsula la creación del modelo. Luego, la variable con el modelo se utiliza en el resto del código. Esta es una de las capacidades que ofrece LangChain.

Si necesitas trabajar con otro LLM, deberás modificar esta función para que lo acepte.

In [3]:
# Esta función crea el modelo de lenguaje (LLM) según el backend elegido.
# Si se elige 'openai', se conecta a la API de OpenAI (requiere clave).
# Si se elige 'ollama', usa un modelo local (requiere tener ollama instalado y el modelo descargado).
def make_llm(
    backend: Literal["openai", "ollama"],
    *,
    temperature: float = 0.1,
    openai_model: str = "gpt-4o-mini",
    ollama_model: str = "llama3",
):
    if backend == "openai":
        return ChatOpenAI(model=openai_model, temperature=temperature)  # requiere OPENAI_API_KEY
    if backend == "ollama":
        return ChatOllama(model=ollama_model, temperature=temperature)   # requiere ollama serve + modelo
    raise ValueError("Backend no soportado")

## Configuraciones de las Tools + funciones necesarias

Se configuran las tools que tendra el agente y otras funciones para agregar funcionalidad a modo de ejemplo.

### Creacion BBDD

In [4]:
# Definimos la ruta a la base de datos SQLite.
# Si no existe, se crea una base de ejemplo con una tabla 'gastos' y algunos datos de prueba.
SQLITE_PATH = "demo.db"

# Esta función crea la base y la tabla si no existen, y agrega algunos datos de ejemplo.
def bootstrap_sqlite(path: str = SQLITE_PATH):
    con = sqlite3.connect(path)
    cur = con.cursor()
    cur.execute("""CREATE TABLE IF NOT EXISTS gastos(
        id INTEGER PRIMARY KEY,
        categoria TEXT,
        monto REAL,
        fecha TEXT
    )""")
    # Si la tabla está vacía, agregamos algunos datos de ejemplo.
    cur.execute("SELECT COUNT(*) FROM gastos")
    if cur.fetchone()[0] == 0:
        cur.executemany(
            "INSERT INTO gastos(categoria, monto, fecha) VALUES (?, ?, ?)",
            [
                ("comida", 2500.0, "2025-08-30"),
                ("transporte", 1800.0, "2025-09-01"),
                ("comida", 3200.5, "2025-09-02"),
            ],
        )
    con.commit(); con.close()

# Ejecutamos la función para asegurarnos de que la base y los datos existen.
bootstrap_sqlite()

### Tools

In [5]:
# Herramienta para mostrar el esquema de la base de datos (tablas y columnas).
# Esto ayuda a saber qué datos hay y cómo están organizados.
@tool
def sql_schema(db_path: str = SQLITE_PATH) -> str:
    """Devuelve un resumen del esquema (tablas y columnas) de la base SQLite apuntada por db_path."""
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name")
    tables = [r[0] for r in cur.fetchall()]
    lines = []
    for t in tables:
        cur.execute(f"PRAGMA table_info({t})")
        cols = ", ".join(f"{c[1]} {c[2]}" for c in cur.fetchall())
        lines.append(f"- {t}: {cols}")
    con.close()
    return "Esquema:\n" + ("\n".join(lines) if lines else "(sin tablas)")

# --- SQL: consultas SELECT seguras ---
@tool
def sql_query(query: str, db_path: str = SQLITE_PATH, max_rows: int = 50) -> str:
    """Ejecuta UNA consulta SELECT segura (sin escritura) sobre SQLite y devuelve hasta max_rows filas."""
    q = query.strip().rstrip(";")
    # validaciones simples para impedir escritura / múltiple statements
    if ";" in query:
        return "Por seguridad, solo se permite una sentencia por llamada."
    if not re.match(r"(?is)^\s*select\b", q):
        return "Solo se permiten consultas SELECT."
    forbidden = r"\b(insert|update|delete|drop|alter|attach|create|replace|vacuum|pragma)\b"
    if re.search(forbidden, q, flags=re.I):
        return "Se detectó una palabra clave no permitida para lectura segura."

    try:
        con = sqlite3.connect(db_path)
        cur = con.cursor()
        cur.execute(q)
        rows = cur.fetchmany(max_rows)
        cols = [d[0] for d in cur.description] if cur.description else []
        con.close()
        if not rows:
            return "(sin resultados)"
        # render simple en tabla
        header = " | ".join(cols)
        sep = "-|-".join("-"*len(c) for c in cols)
        body = "\n".join(" | ".join(str(x) for x in r) for r in rows)
        note = "" if len(rows) < max_rows else f"\n… (mostrando primeras {max_rows} filas)"
        return f"{header}\n{sep}\n{body}{note}"
    except Exception as e:
        return f"Error al ejecutar la consulta: {e}"

# --- otras herramientas de ejemplo ---
@tool
def calc(expression: str) -> str:
    """Calculadora básica para expresiones aritméticas."""
    try:
        allowed = set("0123456789+-*/(). %")
        if not set(expression).issubset(allowed):
            return "La expresión contiene caracteres no permitidos."
        return str(eval(expression))
    except Exception as e:
        return f"Error de cálculo: {e}"

@tool
def todo_brainstorm(topic: str) -> str:
    """Ideas rápidas de tareas."""
    items = [
        f"Definir objetivo concreto para: {topic}",
        f"Listar 3 tareas clave sobre: {topic}",
        f"Identificar bloqueo principal en: {topic}",
        f"Estimar esfuerzo y responsables para: {topic}",
    ]
    return "\n".join(f"- {i}" for i in items)

TOOLS = [sql_schema, sql_query, calc, todo_brainstorm]

## Definicion del Promt

In [6]:
# Definimos el prompt (instrucciones) que guiará al agente.
# El prompt le indica al modelo cómo debe razonar, qué formato seguir y que siempre responda en español.
from langchain.prompts import PromptTemplate

template = """You are a helpful agent that MUST ALWAYS answer in Spanish (Rioplatense).

Conversation so far:
{chat_history}

Follow this format exactly:

Question: the user's question
Thought: briefly explain your plan
Action: the tool to use, exactly one of [{tool_names}]
Action Input: the input for the tool
Observation: the result of the tool
... (repeat Thought/Action/Action Input/Observation as needed)
Final Answer: your concise answer to the user in Spanish

Rules:
- If you choose an Action, DO NOT write 'Final Answer' in the same turn.
- When you choose an Action, respond ONLY with:
  Action: <tool name>
  Action Input: <input>
- Do NOT invent Observations; only write Observation after a tool run.
- If no tool is needed, go directly to Final Answer (in Spanish).

Available tools:
{tools}

Question: {input}
{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(template)

## Creacion del `Agente` y del `Ejecutor`

In [7]:
# Creamos el agente ReAct y el ejecutor que se encargará de interactuar con el usuario y las herramientas.
# El agente sigue el formato definido en el prompt y puede usar las herramientas que definimos antes.
from langchain.agents import create_react_agent, AgentExecutor

llm = make_llm(BACKEND)

react_agent = create_react_agent(
    llm=llm,
    tools=TOOLS,
    prompt=prompt,
)

executor = AgentExecutor.from_agent_and_tools(
    agent=react_agent,
    tools=TOOLS,
    verbose=True,
    # Si el LLM se “desformatea”, devolvé un mensaje amigable en vez de ciclar:
    handle_parsing_errors="Perdón, hubo un error de formato; respondo directo en español.",
    max_iterations=4,   # límite de pasos para evitar loops
)

### Incorporamos Memoria

In [8]:
# --- Memoria por sesión ---
# Usamos un diccionario para guardar el historial de cada sesión.
_store = {}  # dict en memoria: session_id -> ChatMessageHistory

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in _store:
        _store[session_id] = ChatMessageHistory()
    return _store[session_id]

# Envolvemos el ejecutor para que recuerde el historial de la conversación.
agent_with_memory = RunnableWithMessageHistory(
    executor,
    get_session_history,
    input_messages_key="input",     # clave que llega al agente
    history_messages_key="chat_history",  # variable que el prompt espera
)

## Sesion + Interaccion 

In [9]:
# Definimos una función para interactuar con el agente usando una sesión identificada.
# La función recibe un mensaje, lo envía al agente y muestra la respuesta.
SESSION_ID = "sesion_demo_1"  # cambiá para sesiones independientes

def ask(msg: str):
    out = agent_with_memory.invoke(
        {"input": msg},
        config={"configurable": {"session_id": SESSION_ID}},
    )
    print("\n--- RESPUESTA ---\n", out["output"])


# Ejemplo de uso

Dejo comentadas algunas preguntas que fui utilizando para probar el agente.

In [10]:

# --- ejemplo de uso ---
# ask("Si ahorro $200 por mes durante 18 meses y recibo además un bono único de $750, ¿cuánto junto en total?")

# --- ejemplo de uso 2---
# # 1) Mirar el esquema SQLite
# ask("Mostrame el esquema de la base.")
# # 2) Pregunta con consulta SQL (usa memoria del punto anterior)
# ask("¿Cuánto gasté en 'comida' en total? Podés consultar en SQLite.")
# # 3) Otra consulta con continuidad
# ask("Listá las filas de gastos ordenadas por fecha descendente (máximo 5).")

In [11]:
ask("Si ahorro $200 por mes durante 18 meses y recibo además un bono único de $750, ¿cuánto junto en total?")




> Entering new AgentExecutor chain...
Action: calc
Action Input: 200 * 18 + 7504350Final Answer: Juntarías un total de $4350.

> Finished chain.

--- RESPUESTA ---
 Juntarías un total de $4350.


In [12]:
SESSION_ID = "sesion_demo_2"
ask("Mostrame el esquema de la base.")



> Entering new AgentExecutor chain...
Action: sql_schema
Action Input: demo.dbEsquema:
- gastos: id INTEGER, categoria TEXT, monto REAL, fecha TEXTFinal Answer: El esquema de la base de datos incluye una tabla llamada "gastos" con las siguientes columnas: id (INTEGER), categoria (TEXT), monto (REAL) y fecha (TEXT).

> Finished chain.

--- RESPUESTA ---
 El esquema de la base de datos incluye una tabla llamada "gastos" con las siguientes columnas: id (INTEGER), categoria (TEXT), monto (REAL) y fecha (TEXT).


In [13]:
ask("¿Cuánto gasté en 'comida' en total? Podés consultar en SQLite.")



> Entering new AgentExecutor chain...
Thought: Necesito realizar una consulta SQL para sumar los montos de la categoría 'comida' en la tabla de gastos. 
Action: sql_query
Action Input: "SELECT SUM(monto) FROM gastos WHERE categoria = 'comida'"SUM(monto)
----------
5700.5Action: sql_query  
Action Input: "SELECT SUM(monto) FROM gastos WHERE categoria = 'comida'"  SUM(monto)
----------
5700.5Final Answer: Gastaste un total de 5700.5 en 'comida'.

> Finished chain.

--- RESPUESTA ---
 Gastaste un total de 5700.5 en 'comida'.


In [14]:
SESSION_ID = "sesion_demo_3"
ask("¿Cuánto gasté en 'comida' en total? Podés consultar en SQLite.")



> Entering new AgentExecutor chain...
Thought: Necesito consultar la base de datos para obtener el total de gastos en 'comida'. Primero, revisaré el esquema de la base de datos para identificar las tablas y columnas relevantes.  
Action: sql_schema  
Action Input: demo.db  Esquema:
- gastos: id INTEGER, categoria TEXT, monto REAL, fecha TEXTAhora que tengo el esquema, puedo ver que la tabla "gastos" tiene las columnas "categoria" y "monto", que son relevantes para calcular el total de gastos en 'comida'. Procederé a realizar la consulta para sumar los montos de la categoría 'comida'.  
Action: sql_query  
Action Input: SELECT SUM(monto) FROM gastos WHERE categoria = 'comida';  Por seguridad, solo se permite una sentencia por llamada.Action: sql_query  
Action Input: SELECT SUM(monto) FROM gastos WHERE categoria = 'comida';  Por seguridad, solo se permite una sentencia por llamada.Action: sql_query  
Action Input: SELECT SUM(monto) AS total_comida FROM gastos WHERE categoria = 'comida